In [0]:
%sql
SELECT * FROM `emissions`.`default`.`emissions_data`;

state_id,state_abbr,county_state_name,county_id,county_name,latitude,longitude,consolidated_city-county,population,population_cohort,employment,employment_cohort,doe_climate_zone,cohort,consumption (MWh),expenditures in Millions,consumption (MWh/capita),utility customers,GHG emissions mtons CO2e,consumption (TcF),consumption (TcF/capita),occuped housing units,buildings,area (sq. ft.),consumption (gallons),consumption (gallons/capita),vehicle miles traveled (miles),vehicle miles traveled (miles/capita),establishments
1,AL,"Autauga County, AL",1001,Autauga County,32.532237,-86.64644,null,55049,7,21580,5,3,30705,"315,398","39,864",6,"22,120","156,670","316,920",6,"20,800",null,null,"25,332,114",460,"740,266,187","13,447",null
1,AL,"Baldwin County, AL",1003,Baldwin County,30.659218,-87.746067,null,199510,11,122682,9,2,21109,"1,064,843","126,826",5,"127,891","528,947","273,012",1,"75,149",null,null,"116,066,055",582,"3,239,817,322","16,239",null
1,AL,"Barbour County, AL",1005,Barbour County,31.870253,-85.405104,null,26614,6,13613,4,3,30604,"136,098","17,085",5,"11,494","67,605","40,062",2,"9,122",null,null,"14,909,195",560,"391,131,704","14,696",null
1,AL,"Bibb County, AL",1007,Bibb County,33.015893,-87.127148,null,22572,5,6699,2,3,30502,"117,401","14,858",5,"8,738","58,318","99,056",4,"7,048",null,null,"11,806,243",523,"320,513,006","14,200",null
1,AL,"Blount County, AL",1009,Blount County,33.977358,-86.56644,null,57704,8,12557,4,3,30804,"328,466","41,577",6,"23,227","177,690","193,482",3,"20,619",null,null,"23,520,539",408,"673,272,458","11,668",null
1,AL,"Bullock County, AL",1011,Bullock County,32.101759,-85.717261,null,10552,3,3422,1,3,30301,"53,059","6,709",5,"4,345","26,357","31,638",3,"3,556",null,null,"6,186,642",586,"167,836,840","15,906",null
1,AL,"Butler County, AL",1013,Butler County,31.751667,-86.681969,null,20280,5,12211,4,3,30504,"117,812","14,944",6,"9,655","58,522","32,256",2,"7,675",null,null,"22,568,262","1,113","689,255,139","33,987",null
1,AL,"Calhoun County, AL",1015,Calhoun County,33.771706,-85.822513,null,115883,9,71664,8,3,30908,"648,757","82,123",6,"51,932","350,959","788,725",7,"45,071",null,null,"53,512,407",462,"1,453,111,884","12,539",null
1,AL,"Chambers County, AL",1017,Chambers County,32.915504,-85.394032,null,34018,6,12279,4,3,30604,"207,888","26,283",6,"16,463","103,266","153,560",5,"13,851",null,null,"20,775,028",611,"566,425,834","16,651",null
1,AL,"Cherokee County, AL",1019,Cherokee County,34.069515,-85.654242,null,25897,6,7709,3,3,30603,"162,897","20,819",6,"15,835","88,122","86,644",3,"10,999",null,null,"13,580,194",524,"362,839,833","14,011",null


In [0]:
%sql
SELECT 
    latitude,
    longitude,
    `GHG emissions mtons CO2e`as emissions
FROM `emissions`.`default`.`emissions_data`;

latitude,longitude,emissions
32.532237,-86.64644,"156,670"
30.659218,-87.746067,"528,947"
31.870253,-85.405104,"67,605"
33.015893,-87.127148,"58,318"
33.977358,-86.56644,"177,690"
32.101759,-85.717261,"26,357"
31.751667,-86.681969,"58,522"
33.771706,-85.822513,"350,959"
32.915504,-85.394032,"103,266"
34.069515,-85.654242,"88,122"


In [0]:
%sql
SELECT 
    county_state_name,
    population,
    CAST(REPLACE(`GHG emissions mtons CO2e`, ',', '') AS double) / population AS emissions_per_person
FROM `emissions`.`default`.`emissions_data`
ORDER BY emissions_per_person DESC
LIMIT 10;

county_state_name,population,emissions_per_person
"Thayer County, NE",5163,5.94944799535154
"Nelson County, ND",3032,5.106530343007916
"Jefferson County, NE",7354,4.999048137068262
"Steele County, ND",1969,4.990858303707466
"Fillmore County, NE",5676,4.774136715997181
"Traill County, ND",8075,4.744767801857585
"Saline County, NE",14356,4.691696851490666
"Butler County, MO",42887,4.552195303938256
"Maries County, MO",8987,4.541671302993213
"Griggs County, ND",2311,4.521852012115967


In [0]:
%sql
SELECT 
    state_abbr,
    sum(CAST(
        REPLACE(
            `GHG emissions mtons CO2e`, ',', '') AS double
            )) AS Total_emissions
FROM `emissions`.`default`.`emissions_data`
GROUP BY state_abbr
ORDER BY Total_emissions DESC
LIMIT 10;

state_abbr,Total_emissions
TX,5.9016307E7
FL,4.711017E7
OH,2.7792534E7
IL,2.6121703E7
GA,2.4460734E7
MO,2.1245196E7
CA,2.047053E7
TN,2.0463441E7
PA,2.0094423E7
NC,1.9334706E7


In [0]:
%sql
WITH state_emissions AS (
    SELECT 
        state_abbr,
        SUM(CAST(REPLACE(`GHG emissions mtons CO2e`, ',', '') AS double)) AS total_emissions
    FROM `emissions`.`default`.`emissions_data`
    GROUP BY state_abbr
),
country_total AS (
    SELECT SUM(total_emissions) AS total_country_emissions
    FROM state_emissions
)
SELECT 
    FORMAT_NUMBER(s.total_emissions, 0) AS total_emissions_mtons,
    CONCAT(ROUND((s.total_emissions / c.total_country_emissions) * 100, 2), '%') AS pct_of_total_emissions
FROM state_emissions s
CROSS JOIN country_total c
ORDER BY s.total_emissions DESC
LIMIT 10;

total_emissions_mtons,pct_of_total_emissions
"59,016,307",10.49%
"47,110,170",8.38%
"27,792,534",4.94%
"26,121,703",4.64%
"24,460,734",4.35%
"21,245,196",3.78%
"20,470,530",3.64%
"20,463,441",3.64%
"20,094,423",3.57%
"19,334,706",3.44%


In [0]:
%sql
SELECT
    county_state_name,
    population,
    cast(replace(`GHG emissions mtons CO2e`, ',', '') as double) as Total_emissions
from `emissions`.`default`.`emissions_data`
ORDER BY total_emissions DESC
LIMIT 10;


county_state_name,population,Total_emissions
"Maricopa County, AZ",4088549,9809738.0
"Harris County, TX",4434257,9675224.0
"Cook County, IL",5227575,8112382.0
"Miami-Dade County, FL",2664418,5624096.0
"Dallas County, TX",2513054,5432168.0
"Los Angeles County, CA",10057155,5003136.0
"Tarrant County, TX",1947529,4479565.0
"Broward County, FL",1863780,4408545.0
"Clark County, NV",2070153,4241125.0
"Bexar County, TX",1858699,3815856.0
